# 01 - Data audit

Check the UCI Travel Review Ratings data before cleaning or training. Use **MLOP-PJ (WSL)**.

In [1]:
from pathlib import Path
import csv
import hashlib
import io
import json
import platform
import sys
import zipfile
from datetime import datetime, timezone
from urllib.request import urlopen

import numpy as np
import pandas as pd

assert platform.system() == "Linux", "Select the WSL kernel."
assert Path(sys.prefix).name == "mlop-pj", "Select MLOP-PJ (WSL)."
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())
print("Python:", sys.executable)
print("Project:", ROOT)


Python: /home/dew_perawis/.venvs/mlop-pj/bin/python
Project: /mnt/c/Users/SN23341J001185/Documents/MyWork/ICT/ICTYear4_1/MLop/public_teaching_mlaiops/course/MLOP-PJ


## 1. Get the raw file

Source: [UCI](https://archive.ics.uci.edu/dataset/485/travel%2Breview%2Bratings), DOI `10.24432/C5C31Q`, CC BY 4.0. This cell downloads the public CSV and saves its hash. An existing file is reused.

In [2]:
source_url = "https://archive.ics.uci.edu/static/public/485/tarvel%2Breview%2Bratings.zip"
raw_path = ROOT / "data/raw/google_review_ratings.csv"
source_path = ROOT / "reports/data-source.json"
raw_path.parent.mkdir(parents=True, exist_ok=True)
source_path.parent.mkdir(parents=True, exist_ok=True)
downloaded_at = None
if not raw_path.exists():
    with urlopen(source_url, timeout=60) as response:
        archive_bytes = response.read()
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        raw_bytes = archive.read("google_review_ratings.csv")
    with raw_path.open("xb") as output:
        output.write(raw_bytes)
    downloaded_at = datetime.now(timezone.utc).isoformat()
raw_bytes = raw_path.read_bytes()
raw_hash = hashlib.sha256(raw_bytes).hexdigest()
if source_path.exists():
    previous_source = json.loads(source_path.read_text())
    assert previous_source["sha256"] == raw_hash, "Raw file changed. Review it first."
    downloaded_at = previous_source.get("downloaded_at")
source = {
    "dataset": "Travel Review Ratings", "doi": "10.24432/C5C31Q",
    "license": "CC BY 4.0", "source_url": source_url,
    "raw_file": str(raw_path.relative_to(ROOT)), "bytes": len(raw_bytes),
    "sha256": raw_hash, "downloaded_at": downloaded_at,
    "checked_at": datetime.now(timezone.utc).isoformat()
}
source_path.write_text(json.dumps(source, indent=2) + "\n")
print("Raw bytes:", len(raw_bytes))
print("SHA-256:", raw_hash)


Raw bytes: 637001
SHA-256: c07e7751f429ebe6225a77a32e61770dbb7197fd253f667e0434424adbaf7a33


## 2. Check the records

Keep zeros for this audit. Their meaning and cleaning policy are still open. Invalid rows are reported without guessing repairs.

In [3]:
records = list(csv.reader(io.StringIO(raw_bytes.decode("utf-8-sig"))))
header = [name.strip() for name in records[0]]
features = [f"Category {i}" for i in range(1, 25)]
assert len(set(header)) == len(header), "Duplicate column names. Review the CSV."
assert all(name in header for name in features), "Unexpected feature names."
feature_positions = [header.index(name) for name in features]
extra_positions = [i for i in range(1, len(header)) if header[i] not in features]
valid_values, invalid_rows = [], []
for line, row in enumerate(records[1:], start=2):
    reason = None
    if len(row) != len(header):
        reason = "wrong number of fields"
    elif not row[0].strip():
        reason = "missing user ID"
    elif any(row[i].strip() for i in extra_positions):
        reason = "unexpected extra value"
    else:
        try:
            values = np.array([float(row[i]) for i in feature_positions])
            if not np.isfinite(values).all():
                reason = "missing or non-finite rating"
            elif ((values < 0) | (values > 5)).any():
                reason = "rating outside 0 to 5"
        except ValueError:
            reason = "non-numeric rating"
    if reason:
        invalid_rows.append({"csv_line": line, "reason": reason})
    else:
        valid_values.append(values)
X = pd.DataFrame(valid_values, columns=features)
report = {
    "raw_sha256": raw_hash, "rows": len(records) - 1,
    "valid_rows": len(X), "invalid_rows": invalid_rows,
    "feature_count": len(features), "zero_values": int((X == 0).sum().sum()),
    "duplicate_profiles": int(X.duplicated().sum()),
    "zero_policy": "not decided", "cleaned_dataset_saved": False
}
(ROOT / "reports/data-audit.json").write_text(json.dumps(report, indent=2) + "\n")
print(json.dumps(report, indent=2))
X.describe().round(2)


{
  "raw_sha256": "c07e7751f429ebe6225a77a32e61770dbb7197fd253f667e0434424adbaf7a33",
  "rows": 5456,
  "valid_rows": 5454,
  "invalid_rows": [
    {
      "csv_line": 1349,
      "reason": "unexpected extra value"
    },
    {
      "csv_line": 2714,
      "reason": "unexpected extra value"
    }
  ],
  "feature_count": 24,
  "zero_values": 5314,
  "duplicate_profiles": 3,
  "zero_policy": "not decided",
  "cleaned_dataset_saved": false
}


,Category 1,Category 2,Category 3,Category 4,Category 5,Category 6,Category 7,Category 8,Category 9,Category 10,...,Category 15,Category 16,Category 17,Category 18,Category 19,Category 20,Category 21,Category 22,Category 23,Category 24
count,5454.00,5454.00,5454.00,5454.00,5454.00,5454.00,5454.00,5454.00,5454.00,5454.00,...,5454.00,5454.00,5454.00,5454.00,5454.00,5454.00,5454.00,5454.00,5454.00,5454.00
mean,1.46,2.32,2.49,2.80,2.96,2.89,3.35,2.54,3.13,2.83,...,2.21,1.19,0.95,0.82,0.97,1.00,0.97,1.75,1.53,1.56
std,0.83,1.42,1.25,1.31,1.34,1.28,1.41,1.11,1.36,1.31,...,1.72,1.11,0.97,0.95,1.20,1.19,0.93,1.60,1.32,1.17
min,0.00,0.00,0.00,0.83,1.12,1.11,1.12,0.86,0.84,0.81,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.92,1.36,1.54,1.73,1.77,1.79,1.93,1.62,1.80,1.64,...,0.86,0.69,0.58,0.53,0.52,0.54,0.57,0.74,0.79,0.88
50%,1.34,1.91,2.06,2.46,2.67,2.68,3.23,2.17,2.80,2.68,...,1.33,0.80,0.74,0.69,0.69,0.69,0.76,1.03,1.07,1.29
75%,1.81,2.69,2.74,4.10,4.31,3.84,5.00,3.19,5.00,3.53,...,4.44,1.16,0.91,0.84,0.86,0.86,1.00,2.07,1.56,1.66
max,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00,...,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00


## Reviewed result

5,456 raw rows, 5,454 valid rows, 2 invalid rows, 5,314 zero values, and 3 duplicate profiles. Continue to 02 to quarantine the invalid rows and split profiles safely. Zero meaning is still unknown.